### Things to do

- add in psi to the training situation?
- reducing the learning rate dynamically perhaps
- Guan et al group convolutional neural networks to hard-constrain rotational invariance
(translation invariance for beta plane?)
- random forests are possible or not? they preserve invarainces present in the training data. 
- implementation of different discretisations?
- memory state for all models to have prior snapshots in training
- do wind forcing
- GAN network maybe
- fix extend_traj part of dataloading
- remove discard cond based on cfl - i think don't roll out loss, just do updates and record cfl, if it gets crazy raise warning at end. The training data should be bulletproof on entry, so issues lie with generation if not great. 
- add support for nz>2
- add the ZB20 analytic (?) closure in too
- figure out how to include different steps in a curriculum if the models are trained on different sizes
- incorporate the (good) validation statistics into diagnostics

In [12]:
%run ../scripts/profile_jax_bottlenecks --model-type cnn --nx 32 --steps 40 --batch-size 10 --warmup 3 --repeats 10 --output bottleneck-report.json

2026-10-19 18:52:02,887 INFO model.ML.architectures.build_model: Building closure 'cnn' with arch params: {'learning_rate': 0.0001, 'loss': 'mae', 'nlayers': 10, 'width': 64, 'kernel_size': 5, 'activation': 'relu', 'closure_scale': 0.2}
2026-10-19 18:52:05,601 INFO model.ML.utils.loss: Using combined loss functions: ['mae']


[profile] physics
[physics] lower=0.061s compile=0.115s first=0.001s steady-med=0.000130s compile/steady=888.0x
[profile] closure
[closure] lower=0.047s compile=0.158s first=0.034s steady-med=0.029840s compile/steady=5.3x
[profile] rollout
[rollout] lower=0.267s compile=1.916s first=2.688s steady-med=2.647559s compile/steady=0.7x
[profile] train
[train] lower=1.808s compile=3.090s first=190.085s steady-med=183.488116s compile/steady=0.0x

=== JAX bottleneck summary ===
Backend: cpu | JAX: 0.8.0 | Grid: 32x32 (nz=1) | rollout intervals: 40 | batch: 10
Workload         Lower    Compile  First run  Warm median  Compile/warm   Temp memory         FLOPs*
----------------------------------------------------------------------------------------------------
physics         0.061s     0.115s     0.001s    0.000130s        888.0x      20.5 KiB       866079.0
closure         0.047s     0.158s     0.034s    0.029840s          5.3x       3.7 MiB   1685455872.0
rollout         0.267s     1.916s     2

In [ ]:
%run ../run.py

2026-10-19 19:35:41,213 INFO model.ML.utils.loss: Using combined loss functions: ['maddison']
2026-10-19 19:35:41,216 INFO __main__: Requested device: cpu, using device: CPU
2026-10-19 19:36:53,383 INFO __main__: Fine timestep is 2e+03s and coarsened timestep is 3.2e+04s. Model spinup for 20.00 eddy turnover times (~1.9e+04 high-res steps). Training horizon is 2000 low-res steps (~740.74 days). Validation plotting window is 250 steps (~92.59 days).
2026-10-19 19:36:53,439 INFO __main__: Found existing data with matching parameters at u:\Users\Thomas\Documents\Newcastle\QG\QG\data\data_hr512_nx32_08, loading trajectories from there.
2026-10-19 19:36:53,482 INFO __main__: Found existing fno closure with matching parameters at u:\Users\Thomas\Documents\Newcastle\QG\QG\saved_closures\fno\512_to_32\01, attempting to load checkpoint.
2026-10-19 19:36:53,705 INFO model.ML.architectures.build_model: Building closure 'fno' with arch params: {'learning_rate': 1e-05, 'loss': 'maddison', 'width': 